# DUC-03 - Q0 (A3 gốc) vs Q1 (A3 + quality prior) trên Kaggle GPU

Notebook của Đức (nhánh `vinhduc`), theo M0 đã khóa trong `results/contribution_study/m0_locked.json` và `docs/research/contribution-study.md`.

Yêu cầu: Accelerator **GPU**, Internet **On** (để clone repo public), và **Add Input** là bộ dữ liệu Kaggle chứa đúng bốn file output của notebook 01:
`gmdcsa24_pose_32.npz`, `gmdcsa24_features_32.npz`, `gmdcsa24_pose_manifest.csv`, `gmdcsa24_cross_subject_folds.csv`.

Thứ tự bắt buộc, mỗi bước dừng khi FAIL: guard hash input → validate pipeline → phân bố q (chỉ mô tả input) → test phần mềm → smoke thật → **full** → audit Q0→Q1 → reload từng checkpoint → phân tích DUC-04 (khóa trong addendum M0 ngày 18/11). Không sửa cấu hình theo kết quả outer-test. Không dùng output smoke làm kết quả.

In [ ]:
# Commit code đã khóa trong m0_locked.json. Đổi giá trị này = đổi phương án: phải cập nhật M0 trước khi chạy full.
REPO = "https://github.com/aBaoN9/me-stgcn-ta-fall-detection.git"
BRANCH = "vinhduc"
COMMIT = "c72f33a547d3f130d3768611c08949cf45c37b56"
RUN_NAME = "q0_q1_full"  # cố định để resume phiên khác ngày vẫn đúng run

import os, subprocess, sys
from pathlib import Path
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
CODE = WORK / "repo"
if not CODE.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO, str(CODE)], check=True)
subprocess.run(["git", "-C", str(CODE), "checkout", "--detach", COMMIT], check=True)
print(subprocess.run(["git", "-C", str(CODE), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout)
os.chdir(CODE)
sys.path.insert(0, str(CODE))

In [ ]:
import json, platform
import numpy, pandas, sklearn, torch
print(platform.python_version(), "torch", torch.__version__, "numpy", numpy.__version__, "pandas", pandas.__version__, "sklearn", sklearn.__version__)
print("CUDA", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
locked = json.loads(Path("results/contribution_study/m0_locked.json").read_text(encoding="utf-8"))
from training.contribution_study.runner import code_hash, make_config
assert code_hash() == locked["code_hash"], "Code differs from the locked M0; stop"
assert make_config("full") == locked["config"], "Config differs from the locked M0; stop"
print("M0 lock OK:", locked["status"], locked["code_hash"][:12])

In [ ]:
def find_input_root():
    hits = sorted(Path("/kaggle/input").rglob("gmdcsa24_pose_32.npz")) if Path("/kaggle/input").exists() else []
    if len(hits) != 1:
        raise FileNotFoundError(f"Need exactly one gmdcsa24_pose_32.npz under /kaggle/input, found {hits}")
    return hits[0].parent
INPUT_ROOT = find_input_root()
from training.component_ablation.features import load_inputs
from training.contribution_study.runner import verify_inputs
data = load_inputs(INPUT_ROOT)
print("Input guard:", verify_inputs(data, make_config("full")))  # raises unless the four hashes match 07
print({k: v["sha256"][:12] for k, v in data["provenance"].items()})

In [ ]:
def run(*args):
    print("$", " ".join(map(str, args)), flush=True)
    subprocess.run([sys.executable, "-m", *map(str, args)], check=True)

RUN = WORK / RUN_NAME
PRE = WORK / (RUN_NAME + "_precheck")
run("training.component_ablation.validate_pipeline", "--input-root", INPUT_ROOT)
run("training.contribution_study.quality_report", "--input-root", INPUT_ROOT, "--output", PRE / "quality")
run("training.contribution_study.check_quality_pooling")
run("training.contribution_study.check_review_fixes")
run("training.contribution_study.check_analysis")

Smoke thật: 1 seed, 2 epoch, chỉ kiểm phần mềm trên dữ liệu thật. **Không phải kết quả.**

In [ ]:
SMOKE = PRE / "smoke_real"
run("training.contribution_study.runner", "--input-root", INPUT_ROOT, "--mode", "smoke", "--output", SMOKE)
run("training.contribution_study.evaluate_models", "--run", SMOKE, "--input-root", INPUT_ROOT, "--device", "cpu")
print(json.loads((SMOKE / "completion.json").read_text())["scientific_result"])  # must be False

Full: 2 variant × 3 seed × 4 outer = 24 outer / 96 fits. Nếu phiên Kaggle bị ngắt, chạy lại đúng ô này cùng `RUN_NAME`: runner chỉ bỏ qua fold đã kiểm hợp lệ.

In [ ]:
assert torch.cuda.is_available(), "Enable the Kaggle GPU before the full run"
run("training.contribution_study.runner", "--input-root", INPUT_ROOT, "--mode", "full", "--output", RUN)

In [ ]:
run("training.contribution_study.audit_existing", "--run", RUN, "--output", RUN / "audit")
run("training.contribution_study.evaluate_models", "--run", RUN, "--input-root", INPUT_ROOT, "--device", "cpu")
completion = json.loads((RUN / "completion.json").read_text())
print({k: completion[k] for k in ["complete", "scientific_result", "outer_runs", "fits"]})
print(pandas.read_csv(RUN / "seed_mean_std.csv").query("policy == 'fixed'").T)
run("training.contribution_study.analyze_q0_q1", "--run", RUN, "--input-root", INPUT_ROOT)
print(json.dumps(json.loads((RUN / "analysis/analysis_summary.json").read_text())["q0_reproduction_sanity"], indent=2))

In [ ]:
import shutil
for folder in [RUN, PRE]:
    print(shutil.make_archive(str(folder), "zip", folder))
# Tải hai file zip về, giải nén vào results/contribution_study/runs/<RUN_NAME>/ trên nhánh vinhduc.